In [ ]:
# ============================================================
# ENDES 2025 - INTEGRACIÓN DE MICRODATOS
# Región Puno - Niños de 6 a 35 meses
# ============================================================

import pandas as pd
import numpy as np

print("Proyecto: ENDES 2025 - Anemia infantil en Puno")
print("Etapa: Integración inicial de módulos")

In [ ]:
# ============================================================
# 1. CARGA DE MÓDULOS ENDES 2025
# ============================================================

rech0 = pd.read_csv(
    "RECH0_2025.csv",
    sep=",",
    encoding="latin-1",
    low_memory=False
)

rech6 = pd.read_csv(
    "RECH6_2025.csv",
    sep=",",
    encoding="latin-1",
    low_memory=False
)

print("RECH0:", rech0.shape)
print("RECH6:", rech6.shape)

print("\nPrimeras columnas RECH0:")
print(rech0.columns[:10].tolist())

print("\nPrimeras columnas RECH6:")
print(rech6.columns[:10].tolist())

In [ ]:
# ============================================================
# 2. LIMPIEZA DE NOMBRES DE COLUMNAS - ENDES 2025
# ============================================================

rech0.columns = (
    rech0.columns
    .str.replace("\ufeff", "", regex=False)
    .str.replace("ï»¿", "", regex=False)
    .str.strip()
)

rech6.columns = (
    rech6.columns
    .str.replace("\ufeff", "", regex=False)
    .str.replace("ï»¿", "", regex=False)
    .str.strip()
)

print("Primeras columnas RECH0:")
print(rech0.columns[:10].tolist())

print("\nPrimeras columnas RECH6:")
print(rech6.columns[:10].tolist())

In [ ]:
# ============================================================
# 3. VERIFICACIÓN DEL AÑO - ENDES 2025
# ============================================================

print("Años encontrados en RECH0:")
print(rech0["ID1"].value_counts(dropna=False))

print("\nAños encontrados en RECH6:")
print(rech6["ID1"].value_counts(dropna=False))

In [ ]:
# ============================================================
# 4. VERIFICACIÓN DE LA LLAVE HHID - ENDES 2025
# ============================================================

print("¿HHID existe en RECH0?", "HHID" in rech0.columns)
print("¿HHID existe en RECH6?", "HHID" in rech6.columns)

print("\nCantidad de registros:")
print("RECH0:", len(rech0))
print("RECH6:", len(rech6))

print("\nCantidad de HHID únicos:")
print("RECH0:", rech0["HHID"].nunique())
print("RECH6:", rech6["HHID"].nunique())

print("\nDuplicados de HHID en RECH0:")
print(rech0["HHID"].duplicated().sum())

In [ ]:
# ============================================================
# 5. INTEGRACIÓN DE RECH6 + RECH0 - ENDES 2025
# ============================================================

endes_2025 = pd.merge(
    rech6,
    rech0,
    on="HHID",
    how="left",
    validate="many_to_one",
    suffixes=("_RECH6", "_RECH0")
)

print("✅ Integración realizada correctamente.")

print("\nDimensión RECH6 antes de integrar:")
print(rech6.shape)

print("\nDimensión de la base integrada:")
print(endes_2025.shape)

In [ ]:
# ============================================================
# 5.1 VERIFICACIÓN DE COINCIDENCIAS
# ============================================================

sin_coincidencia = endes_2025["HV024"].isna().sum()

print("Registros sin coincidencia con RECH0:")
print(sin_coincidencia)

In [ ]:
# ============================================================
# 5.2 VALIDACIÓN DE LA INTEGRACIÓN
# ============================================================

print("Registros antes de integrar:", len(rech6))
print("Registros después de integrar:", len(endes_2025))

if len(rech6) == len(endes_2025):
    print("✅ No se perdieron registros durante la integración.")
else:
    print("⚠️ Revisar la integración.")

In [ ]:
# ============================================================
# 6.1 VERIFICAR LOS CÓDIGOS DE DEPARTAMENTO
# ============================================================

print("Códigos encontrados en HV024:")

print(
    endes_2025["HV024"]
    .value_counts(dropna=False)
    .sort_index()
)

¿Qué hace?
Primero revisamos los valores que existen en HV024, que identifica el departamento.

In [ ]:
# ============================================================
# 6.2 CONTAR REGISTROS DE PUNO
# ============================================================

registros_puno_2025 = (endes_2025["HV024"] == 21).sum()

print("Registros correspondientes a Puno:")
print(registros_puno_2025)

In [ ]:
# ============================================================
# 6.3 FILTRAR REGIÓN PUNO
# ============================================================

puno_2025 = endes_2025[
    endes_2025["HV024"] == 21
].copy()

print("Base filtrada para Puno:")
print(puno_2025.shape)

Y finalmente validamos:

In [ ]:
# ============================================================
# 6.4 VALIDACIÓN DEL FILTRO
# ============================================================

print("Valores de HV024 después del filtro:")
print(puno_2025["HV024"].unique())

if puno_2025["HV024"].nunique() == 1 and puno_2025["HV024"].iloc[0] == 21:
    print("✅ La base contiene únicamente registros de Puno.")
else:
    print("⚠️ Revisar el filtro.")

In [ ]:
# ============================================================
# 7.1 FILTRO DE EDAD: 6 A 35 MESES - ENDES 2025
# ============================================================

puno_6_35_2025 = puno_2025[
    (puno_2025["HC1"] >= 6) &
    (puno_2025["HC1"] <= 35)
].copy()

print("Niños de 6 a 35 meses en Puno:")
print(puno_6_35_2025.shape)

Este código selecciona únicamente a los niños cuya edad está entre 6 y 35 meses, usando la variable HC1.

In [ ]:
# ============================================================
# 7.2 VALIDACIÓN DEL RANGO DE EDAD
# ============================================================

print("Edad mínima:", puno_6_35_2025["HC1"].min(), "meses")
print("Edad máxima:", puno_6_35_2025["HC1"].max(), "meses")

print("\nCantidad total de niños seleccionados:")
print(len(puno_6_35_2025))

In [ ]:
# ============================================================
# 7.3 COMPROBACIÓN AUTOMÁTICA
# ============================================================

if (
    puno_6_35_2025["HC1"].min() >= 6
    and puno_6_35_2025["HC1"].max() <= 35
):
    print("✅ Todos los registros están entre 6 y 35 meses.")
else:
    print("⚠️ Revisar el filtro de edad.")

In [ ]:
# ============================================================
# 8.1 REVISIÓN DE LA VARIABLE HC57 - ENDES 2025
# ============================================================

print("Valores encontrados en HC57:")

print(
    puno_6_35_2025["HC57"]
    .value_counts(dropna=False)
    .sort_index()
)

¿Qué hace?
Nos permite ver cuántos registros hay en cada categoría de anemia y si existen valores faltantes o códigos distintos.

Ahora revisamos también los porcentajes:

In [ ]:
# ============================================================
# 8.2 DISTRIBUCIÓN PORCENTUAL DE HC57
# ============================================================

porcentaje_hc57_2025 = (
    puno_6_35_2025["HC57"]
    .value_counts(normalize=True, dropna=False)
    .sort_index()
    * 100
).round(2)

print("Porcentaje por categoría de HC57:")
print(porcentaje_hc57_2025)

1 = Anemia grave
2 = Anemia moderada
3 = Anemia leve
4 = Sin anemia

Se realizó una revisión de la variable HC57 con la finalidad de identificar la distribución de los niveles de anemia en niños de 6 a 35 meses de la región Puno durante el año 2025, así como detectar posibles valores faltantes o códigos no válidos antes de construir la variable objetivo.

In [ ]:
# ============================================================
# 9.1 DEPURACIÓN DE HC57 - ENDES 2025
# ============================================================

datos_2025 = puno_6_35_2025[
    puno_6_35_2025["HC57"].isin([1, 2, 3, 4])
].copy()

print("Registros antes de depurar:", len(puno_6_35_2025))
print("Registros después de depurar:", len(datos_2025))

print("\nRegistros eliminados:")
print(len(puno_6_35_2025) - len(datos_2025))

In [ ]:
# ============================================================
# 9.2 CREACIÓN DE target_anemia - ENDES 2025
# ============================================================

datos_2025["target_anemia"] = np.where(
    datos_2025["HC57"].isin([1, 2, 3]),
    1,
    0
)

print("✅ Variable target_anemia creada correctamente.")

1 = Con anemia
0 = Sin anemia

In [ ]:
# ============================================================
# 9.3 DISTRIBUCIÓN DE target_anemia
# ============================================================

print(
    datos_2025["target_anemia"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# 9.4 TABLA FINAL DE ANEMIA - ENDES 2025
# ============================================================

tabla_anemia_2025 = (
    datos_2025["target_anemia"]
    .value_counts()
    .rename(index={
        1: "Con anemia",
        0: "Sin anemia"
    })
    .to_frame("Frecuencia")
)

tabla_anemia_2025["Porcentaje"] = (
    tabla_anemia_2025["Frecuencia"]
    / len(datos_2025)
    * 100
).round(2)

print(tabla_anemia_2025)

In [ ]:
# ============================================================
# 10.1 DISTRIBUCIÓN FINAL DE ANEMIA - ENDES 2025
# ============================================================

tabla_final_2025 = (
    datos_2025["target_anemia"]
    .value_counts()
    .rename(index={
        1: "Con anemia",
        0: "Sin anemia"
    })
    .to_frame("Frecuencia")
)

tabla_final_2025["Porcentaje"] = (
    tabla_final_2025["Frecuencia"]
    / len(datos_2025)
    * 100
).round(2)

print("Distribución final de anemia - ENDES 2025")
print(tabla_final_2025)

In [ ]:
# ============================================================
# 10.2 VALIDACIÓN FINAL
# ============================================================

print("Total de registros finales:", len(datos_2025))
print("Valores de target_anemia:", datos_2025["target_anemia"].unique())

print("\nValores faltantes en target_anemia:")
print(datos_2025["target_anemia"].isna().sum())

In [ ]:
# ============================================================
# 10.3 EXPORTAR ENDES 2025 A EXCEL
# ============================================================

nombre_excel = "ENDES_PUNO_ANEMIA_2025.xlsx"

datos_2025.to_excel(
    nombre_excel,
    index=False
)

print("✅ Archivo Excel generado correctamente:")
print(nombre_excel)

print("\nDimensiones finales:")
print(datos_2025.shape)